# GoRakuDo ComfyUI on Colab（G4 / bf16 本線）
Qwen-Image-2.1 を Google Colab 上の ComfyUI で回すノートブック。**主経路=G4（RTX PRO 6000 Blackwell 96GB）+ bf16フル精度 + Pruna 8step**。
T4/L4/A100 では int8 に自動フォールバックする。
上から順に実行するだけ。詳細は `README-colab.md`。
> ※ Colab実機での実行は未検証（JSON構造・構文・URL到達性のみ検証済み）。


In [ ]:
# ===== セル1: GPU確認 + モード自動選択 =====
import subprocess, re
def detect_gpu():
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader,nounits'],
                         capture_output=True, text=True).stdout.strip()
    print('nvidia-smi:', out)
    line = out.splitlines()[0]
    m = re.search(r'([\\d.]+)\\s*$', line)
    vram_gb = float(m.group(1)) / 1024
    name = line[:m.start()].strip().rstrip(",")
    return name, vram_gb

GPU_NAME, VRAM_GB = detect_gpu()
print(f'GPU: {GPU_NAME} / VRAM: {VRAM_GB:.1f} GB')

MODE_OVERRIDE = ''  # 強制したい時だけ 'bf16' / 'int8' を入れる
if MODE_OVERRIDE in ('bf16', 'int8'):
    MODE = MODE_OVERRIDE
elif VRAM_GB >= 48:   # G4 96GB / A100 80GB -> bf16フル精度
    MODE = 'bf16'
else:                 # T4/L4/A100-40GB等 -> int8
    MODE = 'int8'
# TEはbf16モード&48GB以上でのみbf16版、それ以外はint8版にフォールバック
TE_FILE = 'qwen3vl_8b_bf16.safetensors' if (MODE == 'bf16' and VRAM_GB >= 48) else 'qwen3vl_8b_int8_convrot.safetensors'
COMFY_FLAGS = '--lowvram' if VRAM_GB < 20 else ''
print(f'MODE={MODE} / TE={TE_FILE} / COMFY_FLAGS="{COMFY_FLAGS}"')


In [ ]:
# ===== セル2: ComfyUI インストール（安定版タグ） =====
import os, subprocess
COMFY_TAG = 'v0.37.4'  # 2026-09 時点の安定版。最新に追う場合はタグを更新
%cd /content
if not os.path.isdir('/content/ComfyUI'):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', COMFY_TAG,
                    'https://github.com/comfyanonymous/ComfyUI.git'], check=True)
else:
    print('ComfyUI は取得済み（スキップ）')
%cd /content/ComfyUI
!pip install -q -r requirements.txt
print('install done')


In [ ]:
# ===== セル2b（任意）: ComfyUI-GGUF（Q8 GGUFを使う場合のみ実行） =====
# bf16/int8 の safetensors 経路では不要。このセルは飛ばしてよい。
%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git
print('GGUF node done（ComfyUI再起動後に有効）')


In [ ]:
# ===== セル3: モデルDL（直列・レジューム・スキップ付き、所要時間表示） =====
import os, time, subprocess
%cd /content/ComfyUI
try:
    MODE, TE_FILE
except NameError:
    MODE, TE_FILE = 'bf16', 'qwen3vl_8b_bf16.safetensors'  # セル1未実行時の既定（G4想定）
    print('セル1未実行のため既定 MODE=bf16 を使用')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!pip install -q hf_transfer  # 高速DL。失敗しても下のcurl/wgetが動く

BASE = 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main'
UNET = 'qwen_image_2.1_bf16.safetensors' if MODE == 'bf16' else 'qwen_image_2.1_int8_convrot.safetensors'
FILES = [
    (f'{BASE}/diffusion_models/{UNET}', f'models/diffusion_models/{UNET}'),
    (f'{BASE}/text_encoders/{TE_FILE}', f'models/text_encoders/{TE_FILE}'),
    (f'{BASE}/vae/qwen_image_2.1_vae_bf16.safetensors', 'models/vae/qwen_image_2.1_vae_bf16.safetensors'),
    ('https://huggingface.co/PrunaAI/Pruna-Qwen-Image-2.1/resolve/main/p_qwen_image_2.1_8step_v0.1.safetensors',
     'models/loras/p_qwen_image_2.1_8step_v0.1.safetensors'),
]
t0 = time.time()
for url, dst in FILES:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f'skip (exists): {dst} ({os.path.getsize(dst)/1e9:.2f} GB)')
        continue
    print(f'dl: {url}')
    s = time.time()
    r = subprocess.run(['curl', '-L', '--continue-at', '-', '-o', dst, url])
    if r.returncode != 0 or not os.path.exists(dst) or os.path.getsize(dst) == 0:
        print('  curl失敗 -> wget -c でフォールバック')
        subprocess.run(['wget', '-c', '-O', dst, url], check=True)
    print(f'  done {os.path.getsize(dst)/1e9:.2f} GB / {time.time()-s:.0f}s')
print(f'モデルDL合計: {time.time()-t0:.0f}s')


In [ ]:
# ===== セル4: ワークフロー書き出し（埋め込みJSON -> workflows/） =====
import json, os
try:
    TE_FILE
except NameError:
    TE_FILE = 'qwen3vl_8b_bf16.safetensors'  # セル1未実行時の既定（G4想定）
%cd /content/ComfyUI
# Note: edit/inpaint系は QwenImage21Cache ノード除去済み（aimdo memory compile error回避）。
# bf16系は UNETLoader=bf16 + CLIPLoader=bf16-TE。TEがint8フォールバック時は下で差し替える。
WF_EMBED = {
    't2i_pruna_int8_api.json': json.loads('{"1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}}, "10": {"class_type": "ManualSigmas", "inputs": {"sigmas": "1.0, 0.9333333333333333, 0.8571428571428571, 0.7692307692307693, 0.6666666666666666, 0.5454545454545454, 0.4, 0.2222222222222222, 0.0"}}, "11": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}}, "12": {"class_type": "RandomNoise", "inputs": {"noise_seed": 0}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}}, "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "negative_prompt": "", "prompt": "REPLACE_ME_PROMPT", "resolution": 1024}}, "5": {"class_type": "EmptyLatentImage", "inputs": {"batch_size": 1, "height": 1024, "width": 1024}}, "6": {"class_type": "SamplerCustom", "inputs": {"add_noise": true, "cfg": 1.0, "latent_image": ["5", 0], "model": ["9", 0], "negative": ["4", 1], "noise_seed": 0, "positive": ["4", 0], "sampler": ["11", 0], "sigmas": ["10", 0]}}, "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}}, "8": {"class_type": "SaveImage", "inputs": {"filename_prefix": "gorakudo/cmp_int8_pruna_s8", "images": ["7", 0]}}, "9": {"class_type": "LoraLoaderModelOnly", "inputs": {"lora_name": "p_qwen_image_2.1_8step_v0.1.safetensors", "model": ["1", 0], "strength_model": 1.0}}}'),
    'edit_pruna_int8_api.json': json.loads('{"1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}}, "10": {"class_type": "ComfySwitchNode", "inputs": {"on_false": ["4", 2], "on_true": ["5", 0], "switch": false}}, "11": {"class_type": "LoraLoaderModelOnly", "inputs": {"lora_name": "p_qwen_image_2.1_8step_v0.1.safetensors", "model": ["1", 0], "strength_model": 1.0}}, "13": {"class_type": "ManualSigmas", "inputs": {"sigmas": "1.0, 0.9333333333333333, 0.8571428571428571, 0.7692307692307693, 0.6666666666666666, 0.5454545454545454, 0.4, 0.2222222222222222, 0.0"}}, "14": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}}, "15": {"class_type": "SamplerCustom", "inputs": {"add_noise": true, "cfg": 1.0, "latent_image": ["10", 0], "model": ["11", 0], "negative": ["4", 1], "noise_seed": 0, "positive": ["4", 0], "sampler": ["14", 0], "sigmas": ["13", 0]}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}}, "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "images": {"image_1": ["9", 0]}, "negative_prompt": "", "prompt": "REPLACE_ME_PROMPT", "resolution": 1024, "vae": ["3", 0]}}, "5": {"class_type": "EmptyLatentImage", "inputs": {"batch_size": 1, "height": 1024, "width": 1024}}, "7": {"class_type": "VAEDecode", "inputs": {"samples": ["15", 0], "vae": ["3", 0]}}, "8": {"class_type": "SaveImage", "inputs": {"filename_prefix": "gorakudo/pruna_edit_int8", "images": ["7", 0]}}, "9": {"class_type": "LoadImage", "inputs": {"image": "REPLACE_ME_REF.png"}}}'),
    'inpaint_int8_api.json': json.loads('{"1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}}, "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "images": {"image_1": ["9", 0]}, "negative_prompt": "", "prompt": "REPLACE_ME_PROMPT", "resolution": 1024, "vae": ["3", 0]}}, "6": {"class_type": "KSamplerAdvanced", "inputs": {"add_noise": "enable", "cfg": 1, "end_at_step": 10000, "latent_image": ["12", 0], "model": ["1", 0], "negative": ["4", 1], "noise_seed": 0, "positive": ["4", 0], "return_with_leftover_noise": "disable", "sampler_name": "euler", "scheduler": "simple", "start_at_step": 0, "steps": 25}}, "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}}, "8": {"class_type": "SaveImage", "inputs": {"filename_prefix": "gorakudo/inpaint", "images": ["7", 0]}}, "9": {"class_type": "LoadImage", "inputs": {"image": "REPLACE_ME_BASE.png"}}, "12": {"class_type": "SetLatentNoiseMask", "inputs": {"mask": ["14", 0], "samples": ["13", 0]}}, "13": {"class_type": "VAEEncode", "inputs": {"pixels": ["9", 0], "vae": ["3", 0]}}, "14": {"class_type": "GrowMask", "inputs": {"expand": 6, "mask": ["9", 1], "tapered_corners": true}}}'),
    't2i_bf16_api.json': json.loads('{"1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_bf16.safetensors", "weight_dtype": "default"}}, "10": {"class_type": "ManualSigmas", "inputs": {"sigmas": "1.0, 0.9333333333333333, 0.8571428571428571, 0.7692307692307693, 0.6666666666666666, 0.5454545454545454, 0.4, 0.2222222222222222, 0.0"}}, "11": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}}, "12": {"class_type": "RandomNoise", "inputs": {"noise_seed": 0}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_bf16.safetensors", "type": "qwen_image"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}}, "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "negative_prompt": "", "prompt": "REPLACE_ME_PROMPT", "resolution": 1024}}, "5": {"class_type": "EmptyLatentImage", "inputs": {"batch_size": 1, "height": 1024, "width": 1024}}, "6": {"class_type": "SamplerCustom", "inputs": {"add_noise": true, "cfg": 1.0, "latent_image": ["5", 0], "model": ["9", 0], "negative": ["4", 1], "noise_seed": 0, "positive": ["4", 0], "sampler": ["11", 0], "sigmas": ["10", 0]}}, "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}}, "8": {"class_type": "SaveImage", "inputs": {"filename_prefix": "gorakudo/t2i_bf16_pruna_s8", "images": ["7", 0]}}, "9": {"class_type": "LoraLoaderModelOnly", "inputs": {"lora_name": "p_qwen_image_2.1_8step_v0.1.safetensors", "model": ["1", 0], "strength_model": 1.0}}}'),
    'edit_bf16_api.json': json.loads('{"1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_bf16.safetensors", "weight_dtype": "default"}}, "10": {"class_type": "ComfySwitchNode", "inputs": {"on_false": ["4", 2], "on_true": ["5", 0], "switch": false}}, "11": {"class_type": "LoraLoaderModelOnly", "inputs": {"lora_name": "p_qwen_image_2.1_8step_v0.1.safetensors", "model": ["1", 0], "strength_model": 1.0}}, "13": {"class_type": "ManualSigmas", "inputs": {"sigmas": "1.0, 0.9333333333333333, 0.8571428571428571, 0.7692307692307693, 0.6666666666666666, 0.5454545454545454, 0.4, 0.2222222222222222, 0.0"}}, "14": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}}, "15": {"class_type": "SamplerCustom", "inputs": {"add_noise": true, "cfg": 1.0, "latent_image": ["10", 0], "model": ["11", 0], "negative": ["4", 1], "noise_seed": 0, "positive": ["4", 0], "sampler": ["14", 0], "sigmas": ["13", 0]}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_bf16.safetensors", "type": "qwen_image"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}}, "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "images": {"image_1": ["9", 0]}, "negative_prompt": "", "prompt": "REPLACE_ME_PROMPT", "resolution": 1024, "vae": ["3", 0]}}, "5": {"class_type": "EmptyLatentImage", "inputs": {"batch_size": 1, "height": 1024, "width": 1024}}, "7": {"class_type": "VAEDecode", "inputs": {"samples": ["15", 0], "vae": ["3", 0]}}, "8": {"class_type": "SaveImage", "inputs": {"filename_prefix": "gorakudo/edit_bf16", "images": ["7", 0]}}, "9": {"class_type": "LoadImage", "inputs": {"image": "REPLACE_ME_REF.png"}}}'),
    'inpaint_bf16_api.json': json.loads('{"1": {"class_type": "UNETLoader", "inputs": {"unet_name": "qwen_image_2.1_bf16.safetensors", "weight_dtype": "default"}}, "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_bf16.safetensors", "type": "qwen_image"}}, "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}}, "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "images": {"image_1": ["9", 0]}, "negative_prompt": "", "prompt": "REPLACE_ME_PROMPT", "resolution": 1024, "vae": ["3", 0]}}, "6": {"class_type": "KSamplerAdvanced", "inputs": {"add_noise": "enable", "cfg": 1, "end_at_step": 10000, "latent_image": ["12", 0], "model": ["1", 0], "negative": ["4", 1], "noise_seed": 0, "positive": ["4", 0], "return_with_leftover_noise": "disable", "sampler_name": "euler", "scheduler": "simple", "start_at_step": 0, "steps": 25}}, "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}}, "8": {"class_type": "SaveImage", "inputs": {"filename_prefix": "gorakudo/inpaint_bf16", "images": ["7", 0]}}, "9": {"class_type": "LoadImage", "inputs": {"image": "REPLACE_ME_BASE.png"}}, "12": {"class_type": "SetLatentNoiseMask", "inputs": {"mask": ["14", 0], "samples": ["13", 0]}}, "13": {"class_type": "VAEEncode", "inputs": {"pixels": ["9", 0], "vae": ["3", 0]}}, "14": {"class_type": "GrowMask", "inputs": {"expand": 6, "mask": ["9", 1], "tapered_corners": true}}}'),
}
os.makedirs('workflows', exist_ok=True)
for name, wf in WF_EMBED.items():
    if 'bf16' in name and TE_FILE == 'qwen3vl_8b_int8_convrot.safetensors':
        wf['2']['inputs']['clip_name'] = TE_FILE  # TEフォールバック差し替え
    open(f'workflows/{name}', 'w').write(json.dumps(wf, indent=1))
    print('wrote workflows/' + name)


In [ ]:
# ===== セル4b: GoRakuDo Main Workflow をGUI一覧に置く（任意）（Cacheノード除去済み） =====
# ローカルの「GoRakuDo Main Workflow」をそのまま持ってきたもの。
#   * QwenImage21Cache を2つとも除去して配線し直してある（aimdo memory compile error 回避）
#   * 実行時に MODE に合わせてモデル名を差し替える（bf16 / int8）
import json, os

MAIN_WF = r'''{"id":"e441d2d7-3c79-4a8a-8f6f-e97af47bcd44","revision":0,"last_node_id":60,"last_link_id":252,"nodes":[{"id":1,"type":"UNETLoader","pos":[-582.7405088642553,-958.3862098850879],"size":[262.22222900390625,82],"flags":{},"order":0,"mode":0,"showAdvanced":true,"inputs":[{"localized_name":"unet_name","name":"unet_name","type":"COMBO","widget":{"name":"unet_name"},"link":null},{"localized_name":"\u91cd\u307fdtype","name":"weight_dtype","type":"COMBO","widget":{"name":"weight_dtype"},"link":null}],"outputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"MODEL","type":"MODEL","links":[112]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"UNETLoader"},"widgets_values":["qwen_image_2.1_int8_convrot.safetensors","fp8_e4m3fn_fast"],"widgets_values_named":{"unet_name":"qwen_image_2.1_int8_convrot.safetensors","weight_dtype":"fp8_e4m3fn_fast"}},{"id":2,"type":"CLIPLoader","pos":[-580.5702909670598,-757.096772849308],"size":[241.111083984375,106],"flags":{},"order":1,"mode":0,"showAdvanced":true,"inputs":[{"localized_name":"clip\u540d","name":"clip_name","type":"COMBO","widget":{"name":"clip_name"},"link":null},{"localized_name":"\u30bf\u30a4\u30d7","name":"type","type":"COMBO","widget":{"name":"type"},"link":null},{"localized_name":"\u30c7\u30d0\u30a4\u30b9","name":"device","shape":7,"type":"COMBO","widget":{"name":"device"},"link":null}],"outputs":[{"localized_name":"CLIP","name":"CLIP","type":"CLIP","links":[114]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"CLIPLoader"},"widgets_values":["qwen3vl_8b_int8_convrot.safetensors","qwen_image","default"],"widgets_values_named":{"clip_name":"qwen3vl_8b_int8_convrot.safetensors","type":"qwen_image","device":"default"}},{"id":3,"type":"VAELoader","pos":[-583.1283469357686,-1058.7670420207396],"size":[225,61.94602389835342],"flags":{"collapsed":true},"order":2,"mode":0,"inputs":[{"localized_name":"vae_name","name":"vae_name","type":"COMBO","widget":{"name":"vae_name"},"link":null}],"outputs":[{"localized_name":"VAE","name":"VAE","type":"VAE","links":[115]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"VAELoader"},"widgets_values":["qwen_image_2.1_vae_bf16.safetensors"],"widgets_values_named":{"vae_name":"qwen_image_2.1_vae_bf16.safetensors"}},{"id":4,"type":"TextEncodeQwenImage21","pos":[15.603933339064824,-691.9748106912219],"size":[317.7232606902278,459.47390548427774],"flags":{},"order":19,"mode":0,"inputs":[{"localized_name":"clip","name":"clip","type":"CLIP","link":114},{"localized_name":"image_1","name":"images.image_1","shape":7,"type":"IMAGE","link":null},{"localized_name":"vae","name":"vae","shape":7,"type":"VAE","link":null},{"localized_name":"prompt","name":"prompt","type":"STRING","widget":{"name":"prompt"},"link":null},{"localized_name":"negative_prompt","name":"negative_prompt","type":"STRING","widget":{"name":"negative_prompt"},"link":null},{"localized_name":"resolution","name":"resolution","type":"INT","widget":{"name":"resolution"},"link":null}],"outputs":[{"localized_name":"positive","name":"positive","type":"CONDITIONING","links":[216]},{"localized_name":"negative","name":"negative","type":"CONDITIONING","links":[222]},{"localized_name":"latent","name":"latent","type":"LATENT","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"TextEncodeQwenImage21"},"widgets_values":["1girl, solo, upper body, calm expression, young woman, she holds out one hand toward the viewer with an open palm, inviting them to follow,\nwarm smile, the other arm relaxed at her side, messy wolf cut shoulder length 12\" with vivid royal blue inner colored hair, bright detailed blue eyes, pale skin, wearing classic black turtleneck with plain zipper hoodie jacket, and detailed gothic silver cross necklace, 2D anime illustration, HighRes Anime Artwork, Pixiv Artwork, Pixiv, ultra-detailed, crisp line art, flat matte cel shading, matte texture, plain white background, simple background.","",1024],"widgets_values_named":{"prompt":"1girl, solo, upper body, calm expression, young woman, she holds out one hand toward the viewer with an open palm, inviting them to follow,\nwarm smile, the other arm relaxed at her side, messy wolf cut shoulder length 12\" with vivid royal blue inner colored hair, bright detailed blue eyes, pale skin, wearing classic black turtleneck with plain zipper hoodie jacket, and detailed gothic silver cross necklace, 2D anime illustration, HighRes Anime Artwork, Pixiv Artwork, Pixiv, ultra-detailed, crisp line art, flat matte cel shading, matte texture, plain white background, simple background.","negative_prompt":"","resolution":1024}},{"id":5,"type":"EmptyLatentImage","pos":[-280.73116974675844,-756.7929491079079],"size":[210,106],"flags":{},"order":3,"mode":0,"inputs":[{"localized_name":"\u5e45","name":"width","type":"INT","widget":{"name":"width"},"link":null},{"localized_name":"\u9ad8\u3055","name":"height","type":"INT","widget":{"name":"height"},"link":null},{"localized_name":"\u30d0\u30c3\u30c1\u30b5\u30a4\u30ba","name":"batch_size","type":"INT","widget":{"name":"batch_size"},"link":null}],"outputs":[{"localized_name":"\u6f5c\u5728","name":"LATENT","type":"LATENT","links":[220]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"EmptyLatentImage"},"widgets_values":[1024,1024,1],"widgets_values_named":{"width":1024,"height":1024,"batch_size":1}},{"id":7,"type":"VAEDecode","pos":[-579.0997158078279,-1020.3287431832915],"size":[225,53.963055649063065],"flags":{"collapsed":true},"order":28,"mode":0,"inputs":[{"localized_name":"\u30b5\u30f3\u30d7\u30eb","name":"samples","type":"LATENT","link":221},{"localized_name":"vae","name":"vae","type":"VAE","link":115}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":[122]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"VAEDecode"}},{"id":8,"type":"SaveImage","pos":[417.7241191713216,-861.5989289772382],"size":[320,270],"flags":{"collapsed":true},"order":30,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"images","type":"IMAGE","link":122},{"localized_name":"\u30d5\u30a1\u30a4\u30eb\u540d_\u30d7\u30ec\u30d5\u30a3\u30c3\u30af\u30b9","name":"filename_prefix","type":"STRING","widget":{"name":"filename_prefix"},"link":null}],"outputs":[{"localized_name":"images","name":"images","type":"IMAGE","links":[123]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"SaveImage"},"widgets_values":["gorakudo/cmp_int8_pruna_s8"],"widgets_values_named":{"filename_prefix":"gorakudo/cmp_int8_pruna_s8"}},{"id":9,"type":"LoraLoaderModelOnly","pos":[-284.0259641158463,-1012.5039737062024],"size":[264.4444580078125,82],"flags":{"collapsed":true},"order":18,"mode":0,"inputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"model","type":"MODEL","link":112},{"localized_name":"lora_name","name":"lora_name","type":"COMBO","widget":{"name":"lora_name"},"link":null},{"localized_name":"\u30e2\u30c7\u30eb\u306e\u5f37\u5ea6","name":"strength_model","type":"FLOAT","widget":{"name":"strength_model"},"link":null}],"outputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"MODEL","type":"MODEL","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"LoraLoaderModelOnly"},"widgets_values":["p_qwen_image_2.1_8step_v0.1.safetensors",1],"widgets_values_named":{"lora_name":"p_qwen_image_2.1_8step_v0.1.safetensors","strength_model":1}},{"id":10,"type":"ManualSigmas","pos":[13.100372991547601,-1055.8532371602885],"size":[1110.8517088362453,93.93464257126043],"flags":{"collapsed":false},"order":4,"mode":0,"inputs":[{"localized_name":"\u30b7\u30b0\u30de","name":"sigmas","type":"STRING","widget":{"name":"sigmas"},"link":null}],"outputs":[{"localized_name":"\u30b7\u30b0\u30de","name":"SIGMAS","type":"SIGMAS","links":[218]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"ManualSigmas"},"widgets_values":["1.0, 0.99, 0.98, 0.97, 0.96, 0.89, 0.88, 0.87, 0.86, 0.85, 0.8, 0.79, 0.78, 0.77, 0.76, 0.75, 0.74, 0.73, 0.72, 0.71, 0.7, 0.69, 0.68, 0.67, 0.66, 0.6, 0.35, 0.3, 0.0"],"widgets_values_named":{"sigmas":"1.0, 0.99, 0.98, 0.97, 0.96, 0.89, 0.88, 0.87, 0.86, 0.85, 0.8, 0.79, 0.78, 0.77, 0.76, 0.75, 0.74, 0.73, 0.72, 0.71, 0.7, 0.69, 0.68, 0.67, 0.66, 0.6, 0.35, 0.3, 0.0"}},{"id":11,"type":"KSamplerSelect","pos":[13.677985383457209,-857.2659022603599],"size":[210,58],"flags":{},"order":5,"mode":0,"inputs":[{"localized_name":"\u30b5\u30f3\u30d7\u30e9\u30fc\u540d","name":"sampler_name","type":"COMBO","widget":{"name":"sampler_name"},"link":null}],"outputs":[{"localized_name":"\u30b5\u30f3\u30d7\u30e9\u30fc","name":"SAMPLER","type":"SAMPLER","links":[219]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"KSamplerSelect"},"widgets_values":["euler"],"widgets_values_named":{"sampler_name":"euler"}},{"id":13,"type":"MarkdownNote","pos":[0,1700],"size":[480,320],"flags":{},"order":6,"mode":0,"inputs":[],"outputs":[],"title":"Note: T2I int8+Pruna","properties":{},"widgets_values":["# T2I (int8 + Pruna 8step) \u2014 \u63a8\u5968\u306e\u57fa\u672c\u5f62\n\n\u30c6\u30ad\u30b9\u30c8\u304b\u30891024\u00d71024\u753b\u50cf\u3092\u751f\u6210\u3059\u308b\u3001\u73fe\u884c\u30d9\u30b9\u30c8\u306e\u30b7\u30f3\u30d7\u30eb\u69cb\u6210\u3002\n\n## \u63a8\u5968\u8a2d\u5b9a\uff08\u5909\u3048\u306a\u3044\u3053\u3068\uff09\n- cfg **1.0**\u3001`negative_prompt` \u306f\u57fa\u672c**\u7a7a**\n- \u30b5\u30a4\u30ba **1024\u00d71024**\uff08`EmptyLatentImage`\uff09\n- \u30b7\u30b0\u30de\u306f Pruna 8step \u56fa\u5b9a9\u5024\n  `1.0, 0.9333, 0.8571, 0.7692, 0.6667, 0.5455, 0.4, 0.2222, 0.0`\n- `ManualSigmas` \u3092\u5897\u6e1b\u3057\u305f\u3089 Pruna LoRA \u306e\u524d\u63d0\u304c\u5d29\u308c\u308b\n\n## \u30e2\u30c7\u30eb\n- UNet: `qwen_image_2.1_int8_convrot.safetensors`\n- LoRA: `p_qwen_image_2.1_8step_v0.1.safetensors` (strength **1.0**)\n- TextEncoder: `qwen3vl_8b_int8_convrot.safetensors` (type=qwen_image)\n- VAE: `qwen_image_2.1_vae_bf16.safetensors`\n\n## \u4f7f\u3044\u65b9\n1. `TextEncodeQwenImage21` \u306e `prompt` \u306b\u672c\u6587\u3092\u66f8\u304f\n2. Queue \u3059\u308b\uff08VRAM 12GB \u306e\u305f\u3081**1\u30b8\u30e7\u30d6\u305a\u3064**\uff09\n"],"widgets_values_named":{"text":"# T2I (int8 + Pruna 8step) \u2014 \u63a8\u5968\u306e\u57fa\u672c\u5f62\n\n\u30c6\u30ad\u30b9\u30c8\u304b\u30891024\u00d71024\u753b\u50cf\u3092\u751f\u6210\u3059\u308b\u3001\u73fe\u884c\u30d9\u30b9\u30c8\u306e\u30b7\u30f3\u30d7\u30eb\u69cb\u6210\u3002\n\n## \u63a8\u5968\u8a2d\u5b9a\uff08\u5909\u3048\u306a\u3044\u3053\u3068\uff09\n- cfg **1.0**\u3001`negative_prompt` \u306f\u57fa\u672c**\u7a7a**\n- \u30b5\u30a4\u30ba **1024\u00d71024**\uff08`EmptyLatentImage`\uff09\n- \u30b7\u30b0\u30de\u306f Pruna 8step \u56fa\u5b9a9\u5024\n  `1.0, 0.9333, 0.8571, 0.7692, 0.6667, 0.5455, 0.4, 0.2222, 0.0`\n- `ManualSigmas` \u3092\u5897\u6e1b\u3057\u305f\u3089 Pruna LoRA \u306e\u524d\u63d0\u304c\u5d29\u308c\u308b\n\n## \u30e2\u30c7\u30eb\n- UNet: `qwen_image_2.1_int8_convrot.safetensors`\n- LoRA: `p_qwen_image_2.1_8step_v0.1.safetensors` (strength **1.0**)\n- TextEncoder: `qwen3vl_8b_int8_convrot.safetensors` (type=qwen_image)\n- VAE: `qwen_image_2.1_vae_bf16.safetensors`\n\n## \u4f7f\u3044\u65b9\n1. `TextEncodeQwenImage21` \u306e `prompt` \u306b\u672c\u6587\u3092\u66f8\u304f\n2. Queue \u3059\u308b\uff08VRAM 12GB \u306e\u305f\u3081**1\u30b8\u30e7\u30d6\u305a\u3064**\uff09\n"},"color":"#432","bgcolor":"#653"},{"id":14,"type":"PreviewImage","pos":[412.35028178847574,-862.1452541845018],"size":[456.0018195040973,493.264723071022],"flags":{"collapsed":false},"order":32,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"images","type":"IMAGE","link":123}],"outputs":[{"localized_name":"images","name":"images","type":"IMAGE","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"PreviewImage"}},{"id":17,"type":"SamplerCustom","pos":[107.5241165332178,239.56642565638782],"size":[265.52285803574114,241.9033915009988],"flags":{"collapsed":false},"order":27,"mode":0,"showAdvanced":true,"inputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"model","type":"MODEL","link":null},{"localized_name":"\u30dd\u30b8\u30c6\u30a3\u30d6","name":"positive","type":"CONDITIONING","link":241},{"localized_name":"\u30cd\u30ac\u30c6\u30a3\u30d6","name":"negative","type":"CONDITIONING","link":242},{"localized_name":"\u30b5\u30f3\u30d7\u30e9\u30fc","name":"sampler","type":"SAMPLER","link":154},{"localized_name":"\u30b7\u30b0\u30de","name":"sigmas","type":"SIGMAS","link":247},{"localized_name":"\u6f5c\u5728\u753b\u50cf","name":"latent_image","type":"LATENT","link":246},{"localized_name":"\u30ce\u30a4\u30ba\u3092\u8ffd\u52a0","name":"add_noise","type":"BOOLEAN","widget":{"name":"add_noise"},"link":null},{"localized_name":"\u30ce\u30a4\u30ba\u30b7\u30fc\u30c9","name":"noise_seed","type":"INT","widget":{"name":"noise_seed"},"link":null},{"localized_name":"cfg","name":"cfg","type":"FLOAT","widget":{"name":"cfg"},"link":null}],"outputs":[{"localized_name":"\u51fa\u529b","name":"output","type":"LATENT","links":[243]},{"localized_name":"\u30ce\u30a4\u30ba\u9664\u53bb\u51fa\u529b","name":"denoised_output","type":"LATENT","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"SamplerCustom"},"widgets_values":[true,777,"fixed",2],"widgets_values_named":{"add_noise":true,"noise_seed":777,"control_after_generate":"fixed","cfg":2}},{"id":18,"type":"VAEDecode","pos":[-580.5527086576894,85.47054778070346],"size":[140,46],"flags":{"collapsed":true},"order":29,"mode":0,"inputs":[{"localized_name":"\u30b5\u30f3\u30d7\u30eb","name":"samples","type":"LATENT","link":243},{"localized_name":"vae","name":"vae","type":"VAE","link":202}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":[136]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"VAEDecode"}},{"id":19,"type":"SaveImageAdvanced","pos":[411.2361855054103,441.42960833315016],"size":[280.923828125,93],"flags":{"collapsed":true},"order":31,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"images","type":"IMAGE","link":136},{"localized_name":"\u30d5\u30a1\u30a4\u30eb\u540d\u30d7\u30ec\u30d5\u30a3\u30c3\u30af\u30b9","name":"filename_prefix","type":"STRING","widget":{"name":"filename_prefix"},"link":null},{"localized_name":"\u30d5\u30a9\u30fc\u30de\u30c3\u30c8","name":"format","type":"COMFY_DYNAMICCOMBO_V3","widget":{"name":"format"},"link":null},{"localized_name":"\u30d3\u30c3\u30c8\u6df1\u5ea6","name":"format.bit_depth","type":"COMBO","widget":{"name":"format.bit_depth"},"link":null},{"localized_name":"\u5165\u529b\u30ab\u30e9\u30fc\u30b9\u30da\u30fc\u30b9","name":"format.input_color_space","type":"COMBO","widget":{"name":"format.input_color_space"},"link":null}],"outputs":[{"localized_name":"images","name":"images","type":"IMAGE","links":[137]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0"},"widgets_values":["ComfyUI","png","8-bit","sRGB"],"widgets_values_named":{"filename_prefix":"ComfyUI","format":"png","format.bit_depth":"8-bit","format.input_color_space":"sRGB"}},{"id":20,"type":"PreviewImage","pos":[408.646592957349,488.313944064001],"size":[409.4495204728919,428.94547092286075],"flags":{},"order":33,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"images","type":"IMAGE","link":137}],"outputs":[{"localized_name":"images","name":"images","type":"IMAGE","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"PreviewImage"}},{"id":21,"type":"LoadImage","pos":[-583.5203524099188,746.4939299325122],"size":[255.365234375,321.94448706152747],"flags":{},"order":7,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"image","type":"COMBO","widget":{"name":"image"},"link":null},{"localized_name":"\u30a2\u30c3\u30d7\u30ed\u30fc\u30c9\u3059\u308b\u30d5\u30a1\u30a4\u30eb\u3092\u9078\u629e","name":"upload","type":"IMAGEUPLOAD","widget":{"name":"upload"},"link":null}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":null},{"localized_name":"\u30de\u30b9\u30af","name":"MASK","type":"MASK","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"LoadImage"},"widgets_values":["ComfyUI_00020.png","image"],"widgets_values_named":{"image":"ComfyUI_00020.png","upload":"image"}},{"id":22,"type":"UNETLoader","pos":[-586.2232512190647,140.74322366335196],"size":[266.6666564941406,82],"flags":{},"order":8,"mode":0,"showAdvanced":true,"inputs":[{"localized_name":"unet_name","name":"unet_name","type":"COMBO","widget":{"name":"unet_name"},"link":null},{"localized_name":"\u91cd\u307fdtype","name":"weight_dtype","type":"COMBO","widget":{"name":"weight_dtype"},"link":null}],"outputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"MODEL","type":"MODEL","links":[150]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"UNETLoader"},"widgets_values":["qwen_image_2.1_int8_convrot.safetensors","fp8_e4m3fn_fast"],"widgets_values_named":{"unet_name":"qwen_image_2.1_int8_convrot.safetensors","weight_dtype":"fp8_e4m3fn_fast"}},{"id":24,"type":"LoraLoaderModelOnly","pos":[-285.4568932374284,87.12918970617362],"size":[261.111083984375,82],"flags":{"collapsed":true},"order":20,"mode":0,"inputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"model","type":"MODEL","link":150},{"localized_name":"lora_name","name":"lora_name","type":"COMBO","widget":{"name":"lora_name"},"link":null},{"localized_name":"\u30e2\u30c7\u30eb\u306e\u5f37\u5ea6","name":"strength_model","type":"FLOAT","widget":{"name":"strength_model"},"link":null}],"outputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"MODEL","type":"MODEL","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"LoraLoaderModelOnly"},"widgets_values":["p_qwen_image_2.1_8step_v0.1.safetensors",1],"widgets_values_named":{"lora_name":"p_qwen_image_2.1_8step_v0.1.safetensors","strength_model":1}},{"id":25,"type":"CLIPLoader","pos":[-584.5664189922663,342.95488250170064],"size":[242.22222900390625,106],"flags":{},"order":9,"mode":0,"showAdvanced":false,"inputs":[{"localized_name":"clip\u540d","name":"clip_name","type":"COMBO","widget":{"name":"clip_name"},"link":null},{"localized_name":"\u30bf\u30a4\u30d7","name":"type","type":"COMBO","widget":{"name":"type"},"link":null},{"localized_name":"\u30c7\u30d0\u30a4\u30b9","name":"device","shape":7,"type":"COMBO","widget":{"name":"device"},"link":null}],"outputs":[{"localized_name":"CLIP","name":"CLIP","type":"CLIP","links":[175]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"CLIPLoader"},"widgets_values":["qwen3vl_8b_int8_convrot.safetensors","qwen_image","default"],"widgets_values_named":{"clip_name":"qwen3vl_8b_int8_convrot.safetensors","type":"qwen_image","device":"default"}},{"id":26,"type":"EmptyLatentImage","pos":[-283.84203277792926,845.6103281348992],"size":[210,106],"flags":{"collapsed":false},"order":10,"mode":0,"inputs":[{"localized_name":"\u5e45","name":"width","type":"INT","widget":{"name":"width"},"link":null},{"localized_name":"\u9ad8\u3055","name":"height","type":"INT","widget":{"name":"height"},"link":null},{"localized_name":"\u30d0\u30c3\u30c1\u30b5\u30a4\u30ba","name":"batch_size","type":"INT","widget":{"name":"batch_size"},"link":null}],"outputs":[{"localized_name":"\u6f5c\u5728","name":"LATENT","type":"LATENT","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"EmptyLatentImage"},"widgets_values":[1024,1024,1],"widgets_values_named":{"width":1024,"height":1024,"batch_size":1}},{"id":28,"type":"KSamplerSelect","pos":[15.764503341384966,43.28886119523264],"size":[225,61.96181108317927],"flags":{},"order":11,"mode":0,"inputs":[{"localized_name":"\u30b5\u30f3\u30d7\u30e9\u30fc\u540d","name":"sampler_name","type":"COMBO","widget":{"name":"sampler_name"},"link":null}],"outputs":[{"localized_name":"\u30b5\u30f3\u30d7\u30e9\u30fc","name":"SAMPLER","type":"SAMPLER","links":[154]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"KSamplerSelect"},"widgets_values":["euler"],"widgets_values_named":{"sampler_name":"euler"}},{"id":30,"type":"ManualSigmas","pos":[519.0998828213819,243.58639613340625],"size":[676.2437247722538,93.93465160476764],"flags":{"collapsed":false},"order":12,"mode":0,"inputs":[{"localized_name":"\u30b7\u30b0\u30de","name":"sigmas","type":"STRING","widget":{"name":"sigmas"},"link":null}],"outputs":[{"localized_name":"\u30b7\u30b0\u30de","name":"SIGMAS","type":"SIGMAS","links":[247]}],"title":"\u30a2\u30c3\u30d7\u30b9\u30b1\u30fc\u30eb\u30d7\u30ed\u30f3\u30d7\u30c8\u306e\u30de\u30cb\u30e5\u30a2\u30eb\u30b7\u30b0\u30de","properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"ManualSigmas"},"widgets_values":["0.6, 0.35, 0.3, 0.0"],"widgets_values_named":{"sigmas":"0.6, 0.35, 0.3, 0.0"}},{"id":34,"type":"VAEEncode","pos":[-571.7176963432844,511.1707767809114],"size":[140,46],"flags":{"collapsed":false},"order":22,"mode":0,"inputs":[{"localized_name":"\u30d4\u30af\u30bb\u30eb","name":"pixels","type":"IMAGE","link":252},{"localized_name":"vae","name":"vae","type":"VAE","link":204}],"outputs":[{"localized_name":"\u6f5c\u5728","name":"LATENT","type":"LATENT","links":[246]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"VAEEncode"}},{"id":39,"type":"LoadImage","pos":[-867.0405372450714,746.08676703159],"size":[255.365234375,321.9444418939863],"flags":{},"order":13,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"image","type":"COMBO","widget":{"name":"image"},"link":null},{"localized_name":"\u30a2\u30c3\u30d7\u30ed\u30fc\u30c9\u3059\u308b\u30d5\u30a1\u30a4\u30eb\u3092\u9078\u629e","name":"upload","type":"IMAGEUPLOAD","widget":{"name":"upload"},"link":null}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":null},{"localized_name":"\u30de\u30b9\u30af","name":"MASK","type":"MASK","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"LoadImage"},"widgets_values":["ComfyUI_00019.png","image"],"widgets_values_named":{"image":"ComfyUI_00019.png","upload":"image"}},{"id":40,"type":"TextEncodeQwenImage21","pos":[-286.8117156199853,342.12321814442726],"size":[364.9756070598253,448.01179814064153],"flags":{},"order":21,"mode":0,"inputs":[{"localized_name":"clip","name":"clip","type":"CLIP","link":175},{"localized_name":"image_1","name":"images.image_1","shape":7,"type":"IMAGE","link":197},{"localized_name":"image_2","name":"images.image_2","shape":7,"type":"IMAGE","link":null},{"localized_name":"vae","name":"vae","shape":7,"type":"VAE","link":null},{"localized_name":"prompt","name":"prompt","type":"STRING","widget":{"name":"prompt"},"link":null},{"localized_name":"negative_prompt","name":"negative_prompt","type":"STRING","widget":{"name":"negative_prompt"},"link":null},{"localized_name":"resolution","name":"resolution","type":"INT","widget":{"name":"resolution"},"link":null},{"localized_name":"image_3","name":"images.image_3","shape":7,"type":"IMAGE","link":null}],"outputs":[{"localized_name":"positive","name":"positive","type":"CONDITIONING","links":[241]},{"localized_name":"negative","name":"negative","type":"CONDITIONING","links":[242]},{"localized_name":"latent","name":"latent","type":"LATENT","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"TextEncodeQwenImage21"},"widgets_values":["1girl, solo, upper body, bust portrait from front side, calm expression, young woman, slight smile looking to the camera side, messy wolf cut shoulder length 12\\\" with vivid royal blue inner colored hair, bright detailed blue eyes, pale skin, wearing classic black turtleneck with plain zipper hoodie jacket, and detailed gothic silver cross necklace, 2D anime illustration, HighRes Anime Artwork, Pixiv Artwork, Pixiv, ultra-detailed, crisp line art, flat matte cel shading, matte texture, plain white background, simple background.","",1024],"widgets_values_named":{"prompt":"1girl, solo, upper body, bust portrait from front side, calm expression, young woman, slight smile looking to the camera side, messy wolf cut shoulder length 12\\\" with vivid royal blue inner colored hair, bright detailed blue eyes, pale skin, wearing classic black turtleneck with plain zipper hoodie jacket, and detailed gothic silver cross necklace, 2D anime illustration, HighRes Anime Artwork, Pixiv Artwork, Pixiv, ultra-detailed, crisp line art, flat matte cel shading, matte texture, plain white background, simple background.","negative_prompt":"","resolution":1024}},{"id":41,"type":"ImageScale","pos":[18.194432270365667,847.7692265774331],"size":[225,145.92015406391673],"flags":{},"order":14,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"image","type":"IMAGE","link":null},{"localized_name":"\u62e1\u5927\u65b9\u6cd5","name":"upscale_method","type":"COMBO","widget":{"name":"upscale_method"},"link":null},{"localized_name":"\u5e45","name":"width","type":"INT","widget":{"name":"width"},"link":null},{"localized_name":"\u9ad8\u3055","name":"height","type":"INT","widget":{"name":"height"},"link":null},{"localized_name":"\u30af\u30ed\u30c3\u30d7","name":"crop","type":"COMBO","widget":{"name":"crop"},"link":null}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"ImageScale"},"widgets_values":["lanczos",1280,1280,"disabled"],"widgets_values_named":{"upscale_method":"lanczos","width":1280,"height":1280,"crop":"disabled"}},{"id":43,"type":"LoadImage","pos":[-884.4661508786276,346.5676818255598],"size":[255.365234375,321.93180819396497],"flags":{},"order":15,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"image","type":"COMBO","widget":{"name":"image"},"link":null},{"localized_name":"\u30a2\u30c3\u30d7\u30ed\u30fc\u30c9\u3059\u308b\u30d5\u30a1\u30a4\u30eb\u3092\u9078\u629e","name":"upload","type":"IMAGEUPLOAD","widget":{"name":"upload"},"link":null}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":[197,252]},{"localized_name":"\u30de\u30b9\u30af","name":"MASK","type":"MASK","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"LoadImage"},"widgets_values":["cmp_int8_pruna_s8_00084_ (1).png","image"],"widgets_values_named":{"image":"cmp_int8_pruna_s8_00084_ (1).png","upload":"image"}},{"id":47,"type":"VAELoader","pos":[-582.7837997848936,42.128512115703366],"size":[225,61.94602060676894],"flags":{"collapsed":true},"order":16,"mode":0,"inputs":[{"localized_name":"vae_name","name":"vae_name","type":"COMBO","widget":{"name":"vae_name"},"link":null}],"outputs":[{"localized_name":"VAE","name":"VAE","type":"VAE","links":[202,204]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"VAELoader"},"widgets_values":["qwen_image_2.1_vae_bf16.safetensors"],"widgets_values_named":{"vae_name":"qwen_image_2.1_vae_bf16.safetensors"}},{"id":55,"type":"SamplerCustom","pos":[-284.40191177469643,-552.0241385774948],"size":[265.52285803574114,241.9033915009988],"flags":{"collapsed":false},"order":26,"mode":0,"showAdvanced":true,"inputs":[{"localized_name":"\u30e2\u30c7\u30eb","name":"model","type":"MODEL","link":null},{"localized_name":"\u30dd\u30b8\u30c6\u30a3\u30d6","name":"positive","type":"CONDITIONING","link":216},{"localized_name":"\u30cd\u30ac\u30c6\u30a3\u30d6","name":"negative","type":"CONDITIONING","link":222},{"localized_name":"\u30b5\u30f3\u30d7\u30e9\u30fc","name":"sampler","type":"SAMPLER","link":219},{"localized_name":"\u30b7\u30b0\u30de","name":"sigmas","type":"SIGMAS","link":218},{"localized_name":"\u6f5c\u5728\u753b\u50cf","name":"latent_image","type":"LATENT","link":220},{"localized_name":"\u30ce\u30a4\u30ba\u3092\u8ffd\u52a0","name":"add_noise","type":"BOOLEAN","widget":{"name":"add_noise"},"link":null},{"localized_name":"\u30ce\u30a4\u30ba\u30b7\u30fc\u30c9","name":"noise_seed","type":"INT","widget":{"name":"noise_seed"},"link":null},{"localized_name":"cfg","name":"cfg","type":"FLOAT","widget":{"name":"cfg"},"link":null}],"outputs":[{"localized_name":"\u51fa\u529b","name":"output","type":"LATENT","links":[221]},{"localized_name":"\u30ce\u30a4\u30ba\u9664\u53bb\u51fa\u529b","name":"denoised_output","type":"LATENT","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"SamplerCustom"},"widgets_values":[true,777,"fixed",1],"widgets_values_named":{"add_noise":true,"noise_seed":777,"control_after_generate":"fixed","cfg":1}},{"id":59,"type":"ImageBlur","pos":[-597.949765190262,611.2864244208915],"size":[270,82],"flags":{},"order":17,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"image","type":"IMAGE","link":null},{"localized_name":"\u30d6\u30e9\u30fc\u30ec\u30a4\u30c7\u30a3\u30a6\u30b9","name":"blur_radius","type":"INT","widget":{"name":"blur_radius"},"link":null},{"localized_name":"\u30b7\u30b0\u30de","name":"sigma","type":"FLOAT","widget":{"name":"sigma"},"link":null}],"outputs":[{"localized_name":"\u753b\u50cf","name":"IMAGE","type":"IMAGE","links":[250]}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"ImageBlur"},"widgets_values":[1,6],"widgets_values_named":{"blur_radius":1,"sigma":6}},{"id":60,"type":"PreviewImage","pos":[-276.85677573726764,1019.602631741089],"size":[140,26],"flags":{},"order":23,"mode":0,"inputs":[{"localized_name":"\u753b\u50cf","name":"images","type":"IMAGE","link":250}],"outputs":[{"localized_name":"images","name":"images","type":"IMAGE","links":null}],"properties":{"cnr_id":"comfy-core","ver":"0.37.0","Node name for S&R":"PreviewImage"}}],"links":[[112,1,0,9,0,"MODEL"],[114,2,0,4,0,"CLIP"],[115,3,0,7,1,"VAE"],[122,7,0,8,0,"IMAGE"],[123,8,0,14,0,"IMAGE"],[137,19,0,20,0,"IMAGE"],[150,22,0,24,0,"MODEL"],[154,28,0,17,3,"SAMPLER"],[202,47,0,18,1,"VAE"],[204,47,0,34,1,"VAE"],[216,4,0,55,1,"CONDITIONING"],[218,10,0,55,4,"SIGMAS"],[219,11,0,55,3,"SAMPLER"],[220,5,0,55,5,"LATENT"],[221,55,0,7,0,"LATENT"],[222,4,1,55,2,"CONDITIONING"],[241,40,0,17,1,"CONDITIONING"],[242,40,1,17,2,"CONDITIONING"],[243,17,0,18,0,"LATENT"],[246,34,0,17,5,"LATENT"],[247,30,0,17,4,"SIGMAS"],[250,59,0,60,0,"IMAGE"],[252,43,0,34,0,"IMAGE"],[136,18,0,19,0,"IMAGE"],[175,25,0,40,0,"CLIP"],[197,43,0,40,1,"IMAGE"]],"groups":[],"config":{},"extra":{"ds":{"scale":1.0152559799477345,"offset":[765.435160277034,1331.1892693328975]}},"version":0.4}'''

wf = json.loads(MAIN_WF)
UNET_NAME = 'qwen_image_2.1_bf16.safetensors' if MODE == 'bf16' else 'qwen_image_2.1_int8_convrot.safetensors'
for n in wf['nodes']:
    t = n.get('type')
    if t == 'UNETLoader' and n.get('widgets_values'):
        n['widgets_values'][0] = UNET_NAME
    elif t == 'CLIPLoader' and n.get('widgets_values'):
        n['widgets_values'][0] = TE_FILE

paths = [
    '/content/ComfyUI/user/default/workflows/GoRakuDo Main Workflow.json',
    '/content/ComfyUI/user/default/workflows/GoRakuDo - T2I int8+Pruna.json',
]
os.makedirs(os.path.dirname(paths[0]), exist_ok=True)
json.dump(wf, open(paths[0], 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
for p in paths:
    print('wrote:', p)
print()
print('GUIの左サイドバー「ワークフロー」から開けます。')
print('  * モデル名はこのランタイム用に差し替え済み（' + UNET_NAME + '）')
print('  * 触らない設定: cfg=1.0 / ネガティブ空 / 1024 / ManualSigmasのPruna 8step値 / LoRA 1.0')


In [ ]:
# ===== セル5: ComfyUI起動 + cloudflared公開URL発行 =====
import subprocess, time, re, os, urllib.request
try:
    COMFY_FLAGS
except NameError:
    COMFY_FLAGS = ''  # セル1未実行時の既定（G4想定: フラグなし）
%cd /content/ComfyUI
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
    print('cloudflared取得done')
log = open('/content/comfy.log', 'w')
subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', *COMFY_FLAGS.split()],
                 cwd='/content/ComfyUI', stdout=log, stderr=subprocess.STDOUT)
print(f'ComfyUI起動中... FLAGS="{COMFY_FLAGS}" 起動待ち:')
for i in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=5)
        print('ComfyUI up!')
        break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError('ComfyUIが起動しません。/content/comfy.log を確認')
cf = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = cf.stdout.readline()
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", line)
    if m:
        url = m.group(0)
        break
print('\\n公開URL:', url)
print('ブラウザで開いてComfyUI GUIが使える（workflows/ のJSONを読み込んで実行可）')


In [ ]:
# ===== セル6: APIで1枚生成 -> PNG保存 -> ダウンロード（GUI不要） =====
import json, time, urllib.request
from google.colab import files
try:
    MODE
except NameError:
    MODE = 'bf16'  # セル1未実行時の既定（G4想定）

PROMPT = 'a cozy fantasy tavern interior, warm lantern light, anime style, highly detailed'
NEGATIVE = ''  # Pruna 8stepはCFG=1・ネガティブ空が推奨
SEED = 12345
RESOLUTION = 1024  # 1024/1280/1536/2048。G4+bf16なら1536〜2048も実用的
OUTNAME = 'gorakudo_g4_01.png'

WF_NAME = 't2i_bf16_api.json' if MODE == 'bf16' else 't2i_pruna_int8_api.json'
wf = json.load(open(f'/content/ComfyUI/workflows/{WF_NAME}'))
wf['4']['inputs']['prompt'] = PROMPT
wf['4']['inputs']['negative_prompt'] = NEGATIVE
wf['4']['inputs']['resolution'] = RESOLUTION
wf['5']['inputs']['width'] = RESOLUTION
wf['5']['inputs']['height'] = RESOLUTION
for k, n in wf.items():
    if n['class_type'] in ('SamplerCustom', 'KSamplerAdvanced', 'RandomNoise'):
        n['inputs']['noise_seed'] = SEED

t0 = time.time()
req = urllib.request.Request('http://127.0.0.1:8188/prompt',
                             data=json.dumps({'prompt': wf}).encode(),
                             headers={'Content-Type': 'application/json'})
pid = json.load(urllib.request.urlopen(req))['prompt_id']
print('prompt_id:', pid)
while True:
    h = json.load(urllib.request.urlopen(f'http://127.0.0.1:8188/history/{pid}'))
    if pid in h and h[pid].get('status', {}).get('completed'):
        break
    print('  generating...')
    time.sleep(5)
print(f'生成時間: {time.time()-t0:.0f}s')
for nid, nd in h[pid]['outputs'].items():
    for img in nd.get('images', []):
        q = f"filename={img['filename']}&subfolder={img['subfolder']}&type={img['type']}"
        data = urllib.request.urlopen(f'http://127.0.0.1:8188/view?{q}').read()
        open('/content/' + OUTNAME, 'wb').write(data)
        print('saved', OUTNAME, f'{len(data)/1e6:.1f} MB')
        break
    break
files.download('/content/' + OUTNAME)


In [ ]:
# ===== セル7（任意）: バッチ生成 -> zipでダウンロード =====
import json, time, urllib.request, zipfile, glob, os
from IPython.display import Image, display
try:
    MODE
except NameError:
    MODE = 'bf16'

PROMPTS = [
    'a cozy fantasy tavern interior, warm lantern light, anime style, highly detailed',
    'a quiet library dungeon with floating candles, anime style, highly detailed',
]
SAME_PROMPT_N = 0  # >0ならPROMPTS[0]をseed連番でN枚（G4なら10枚でも1〜2分）
BASE_SEED = 100
RESOLUTION = 1024  # 1024/1280/1536/2048
OUTDIR = '/content/batch_out'

os.makedirs(OUTDIR, exist_ok=True)
WF_NAME = 't2i_bf16_api.json' if MODE == 'bf16' else 't2i_pruna_int8_api.json'
base_wf = json.load(open(f'/content/ComfyUI/workflows/{WF_NAME}'))

def gen_one(prompt, seed, outpath):
    wf = json.loads(json.dumps(base_wf))
    wf['4']['inputs']['prompt'] = prompt
    wf['4']['inputs']['negative_prompt'] = ''
    wf['4']['inputs']['resolution'] = RESOLUTION
    wf['5']['inputs']['width'] = RESOLUTION
    wf['5']['inputs']['height'] = RESOLUTION
    for k, n in wf.items():
        if n['class_type'] in ('SamplerCustom', 'KSamplerAdvanced', 'RandomNoise'):
            n['inputs']['noise_seed'] = seed
    req = urllib.request.Request('http://127.0.0.1:8188/prompt',
                                 data=json.dumps({'prompt': wf}).encode(),
                                 headers={'Content-Type': 'application/json'})
    pid = json.load(urllib.request.urlopen(req))['prompt_id']
    while True:
        h = json.load(urllib.request.urlopen(f'http://127.0.0.1:8188/history/{pid}'))
        if pid in h and h[pid].get('status', {}).get('completed'):
            break
        time.sleep(5)
    for nid, nd in h[pid]['outputs'].items():
        for img in nd.get('images', []):
            q = f"filename={img['filename']}&subfolder={img['subfolder']}&type={img['type']}"
            open(outpath, 'wb').write(urllib.request.urlopen(f'http://127.0.0.1:8188/view?{q}').read())
            return

jobs = ([(PROMPTS[0], BASE_SEED + i) for i in range(SAME_PROMPT_N)] if SAME_PROMPT_N > 0
        else [(p, BASE_SEED + i) for i, p in enumerate(PROMPTS)])
t0 = time.time()
for i, (p, s) in enumerate(jobs):
    out = f'{OUTDIR}/batch_{i:02d}_s{s}.png'
    gen_one(p, s, out)
    print(f'[{i+1}/{len(jobs)}] saved {out}')
    display(Image(out, width=384))
print(f'バッチ合計: {time.time()-t0:.0f}s / {len(jobs)}枚')
with zipfile.ZipFile('/content/batch_out.zip', 'w') as z:
    for f in sorted(glob.glob(f'{OUTDIR}/*.png')):
        z.write(f, os.path.basename(f))
from google.colab import files
files.download('/content/batch_out.zip')


In [ ]:
# ===== セル8（任意）: キープアライブ（アイドル切断避けの小技） =====
from IPython.display import Javascript, display
# Note: ブラウザタブを開いたままにする事が条件。確実ではないおまじない。
display(Javascript('setInterval(function(){console.log("grkd-keepalive " + new Date().toISOString());}, 60*1000);'))
print('keepalive timer started（コンソールに60秒毎出力）')
